# 实验流程说明

本项目围绕 **LLM 数学推理评估（GSM8K Best-of-N & ProcessBench）** 展开，采用**冻结编码器（Frozen Encoder）+ 轻量级奖励模型（Reward Head）**的方案，对数学推理轨迹进行评分与重排序，并从数据生成、模型训练、外部评估、内在评估等多个角度分析奖励模型的性能。

整个实验流程如下图所示：

```
Math-Shepherd / GSM8K
        │
        ▼
数据预处理
        │
        ▼
冻结 Qwen2.5-0.5B 提取特征
(precompute_embeddings.py)
        │
        ▼
Embedding Cache
        │
        ▼
训练 Reward Head
(train_from_cache.py)
        │
        ▼
┌───────────────┬────────────────┬─────────────────┐
│               │                │                 │
▼               ▼                ▼                 ▼
Step评估     Single评估      BoN评估       Representation
(eval_step) (eval_single) (eval_bon) (analyze_rep)
```

---

# 第一阶段：数据准备

## 数据集

实验共涉及三类数据。

### ① Math-Shepherd

用于训练 Process Reward Model（PRM）。

主要文件：

```
data/
├── train.jsonl
├── val.jsonl
└── math_shepherd_full.jsonl
```

每条数据格式：

```json
{
    "question": "...",
    "steps": [
        "...",
        "...",
        "..."
    ],
    "labels":[1,1,0]
}
```

其中：

- question：题目
- steps：推理步骤
- labels：每一步是否正确

---

### ② GSM8K

用于生成 Best-of-N 候选轨迹。

生成后得到：

```
data/gsm8k_qwen0.5b_bon16.jsonl
```

每题包含 16 条候选解答。

---

### ③ ProcessBench

用于奖励模型评估。

```
data/processbench_bon_gsm8k.jsonl
```

包含：

- 候选推理轨迹
- label（最终是否正确）

---

# 第二阶段：冻结编码器特征提取

由于 Qwen2.5-0.5B 在整个实验过程中保持冻结，因此仅需进行一次前向传播。

对应文件：

```
encoder.py
precompute_embeddings.py
```

运行命令：

```bash
python precompute_embeddings.py \
    --train_file data/train.jsonl \
    --cache_dir cache/train_clean
```

验证集：

```bash
python precompute_embeddings.py \
    --train_file data/val.jsonl \
    --cache_dir cache/val_clean
```

输出：

```
cache/
├── train_clean/
└── val_clean/
```

之后所有训练均直接读取缓存，不再调用 LLM。

---

# 第三阶段：训练 Reward Head

训练过程中冻结编码器，仅优化奖励头。

对应文件：

```
reward_heads.py
pqm_loss.py
train_from_cache.py
```

支持五种奖励头：

- Linear
- MLP
- CNN
- GRU
- Attention

训练命令：

```bash
python train_from_cache.py \
    --cache_dir cache/train_clean \
    --head gru \
    --epochs 10 \
    --save_path checkpoints/gru_clean.pt
```

模型保存至：

```
checkpoints/

gru_clean.pt
cnn_clean.pt
mlp_clean.pt
linear_full.pt
...
```

同时生成：

```
results/

gru_efficiency.json
cnn_efficiency.json
...
```

记录：

- 参数量
- 峰值显存
- 训练耗时

---

# 第四阶段：Step-level 评估

评估奖励模型是否能够识别正确推理步骤。

对应文件：

```
eval_step_metrics.py
```

运行：

```bash
python eval_step_metrics.py \
    --cache_dir cache/val_clean \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

输出：

```
results/gru_step_metrics.json
```

主要指标：

- Step Accuracy
- Q-value Ranking Accuracy

---

# 第五阶段：Single Trajectory Evaluation

用于验证：

> 奖励模型是否能够判断整条推理是否正确。

对应文件：

```
precompute_eval_embeddings.py

eval_single_from_cache.py
```

首先预计算：

```bash
python precompute_eval_embeddings.py \
    --eval_file data/single_eval.jsonl \
    --cache_dir cache/single_eval
```

随后评估：

```bash
python eval_single_from_cache.py \
    --cache_dir cache/single_eval \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

输出：

```
results/gru_single_metrics.json
```

主要指标：

- Accuracy
- Pairwise Separation

---

# 第六阶段：生成 GSM8K Best-of-N 数据

为了验证 PRM 的重排序能力，需要首先利用生成模型产生多个候选解答。

对应文件：

```
generate_bon_data.py
```

生成模型：

```
Qwen/Qwen2.5-0.5B-Instruct
```

运行：

```bash
python generate_bon_data.py
```

生成：

```
data/gsm8k_qwen0.5b_bon16.jsonl
```

实验配置：

| 参数 | 数值 |
|------|------|
| 数据集 | GSM8K |
| 题目数量 | 1319 |
| 每题采样 | 16 |

---

# 第七阶段：Oracle Evaluation

Oracle 用于计算生成器理论上限。

对应文件：

```
eval_oracle.py
```

运行：

```bash
python eval_oracle.py \
    --eval_file data/gsm8k_qwen0.5b_bon16.jsonl
```

实验结果：

| 指标 | 数值 |
|------|------|
| Oracle@16 | 76.04% |

说明：

如果奖励模型足够优秀，理论上能够达到 76.04%。

---

# 第八阶段：BoN 外部评估

这是整个项目最重要的实验。

对应文件：

```
eval_bon.py
```

运行：

```bash
python eval_bon.py \
    --eval_file data/gsm8k_qwen0.5b_bon16.jsonl \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt \
    --ks 16
```

同时实现了四种聚合策略：

- last
- min
- mean
- sum

实验结果：

| Aggregation | BoN@16 |
|--------------|--------|
| last | 34.04% |
| min | 32.37% |
| sum | 32.22% |
| mean | 31.99% |

说明：

GRU 更依赖最后一步隐藏状态，而无法充分利用完整推理过程。

---

# 第九阶段：ProcessBench 内在评估

为了进一步分析 GRU 排序能力较弱的原因，直接将奖励模型作为二分类器进行测试。

对应文件：

```
eval_intrinsic.py
```

运行：

```bash
python eval_intrinsic.py \
    --eval_file data/processbench_bon_gsm8k.jsonl \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

输出：

```
results/gru_intrinsic.json
```

实验结果：

| 指标 | 数值 |
|------|------|
| ROC-AUC | 0.6139 |
| Accuracy | 0.5000 |

说明：

GRU 对正确轨迹与错误轨迹的区分能力较弱。

---

# 第十阶段：效率评测

对应文件：

```
benchmark.py
```

运行：

```bash
python benchmark.py \
    --model_name Qwen/Qwen2.5-0.5B \
    --head gru \
    --batch_size 48 \
    --seq_len 512 \
    --dataset_size 445000
```

统计：

- GPU 显存
- 吞吐率
- 每 Epoch 时间
- 总训练时间

---

# 第十一阶段：表征分析

对应文件：

```
analyze_representations.py
```

包括：

### Encoder 特征可视化

```bash
python analyze_representations.py \
    --cache_dir cache/val_clean \
    --mode encoder_tsne
```

### Q-value 分布

```bash
python analyze_representations.py \
    --cache_dir cache/val_clean \
    --mode qvalue_dist \
    --head gru \
    --checkpoint checkpoints/gru_clean.pt
```

输出：

```
results/

qvalue_dist_gru.png
```

---

# 第十二阶段：结果汇总

对应文件：

```
summarize_results.py
```

运行：

```bash
python summarize_results.py \
    --results_dir results
```

自动生成：

```
results/

summary.csv

summary.md
```

最终汇总：

- 参数量
- 显存
- 训练耗时
- Step Accuracy
- Pairwise Separation
- BoN@8
- BoN@16
- Oracle
- ROC-AUC

---

# 项目文件对应关系

| 文件 | 功能 |
|------|------|
| `encoder.py` | 冻结 Qwen 编码器 |
| `reward_heads.py` | Linear / MLP / CNN / GRU / Attention 奖励头 |
| `pqm_loss.py` | PQM Ranking Loss |
| `dataset.py` | 数据读取 |
| `precompute_embeddings.py` | 预计算训练/验证 Embedding |
| `precompute_eval_embeddings.py` | 单条评估 Embedding |
| `train_from_cache.py` | 训练奖励头 |
| `benchmark.py` | 训练效率测试 |
| `generate_bon_data.py` | 生成 GSM8K Best-of-N 候选轨迹 |
| `eval_step_metrics.py` | Step-level 评估 |
| `eval_single_from_cache.py` | 单条轨迹评估 |
| `eval_oracle.py` | Oracle 理论上限 |
| `eval_bon.py` | Best-of-N 重排序评估 |
| `eval_intrinsic.py` | ProcessBench 内在评估 |
| `analyze_representations.py` | 表征可视化 |
| `summarize_results.py` | 汇总实验结果 |

In [1]:
from datasets import load_dataset
import random
import json

# 抽取数量（100~200之间任选）
N = 150

# 固定随机种子，保证结果可复现
SEED = 42

# 加载训练集
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 随机抽样
random.seed(SEED)
indices = random.sample(range(len(dataset)), N)

# 保存为 JSONL
with open("dummy_train.jsonl", "w", encoding="utf-8") as f:
    for idx in indices:
        json.dump(dataset[idx], f, ensure_ascii=False)
        f.write("\n")

print(f"Saved {N} samples to dummy_train.jsonl")

f:\anaconda3\envs\node2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Saved 150 samples to dummy_train.jsonl


In [2]:
from datasets import load_dataset
import json

# 加载 Math-Shepherd 数据集（请替换为具体的 HF 仓库名，如 "math-shepherd/..."）
dataset = load_dataset(
    "peiyi9979/Math-Shepherd",
    split="train"
)

# 导出为你代码需要的 jsonl 格式
output_file = "data/math_shepherd_full.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for item in dataset:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
        
print(f"下载完成！共 {len(dataset)} 条数据，已保存至 {output_file}")

KeyboardInterrupt: 